<a href="https://colab.research.google.com/github/Divyanshu8410/NLP/blob/main/24_Fine_Tuning_BERT_for_text_classification_using_Hugging_Face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install transformers torch scikit-learn

In [2]:

# Install libraries:
# pip install transformers torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Simple Dataset
texts = [
    "I love this movie",
    "This is a great product",
    "The food is delicious",
    "I am very happy",
    "Excellent experience",
    "This is amazing",
    "I hate this movie",
    "This product is bad",
    "The food is terrible",
    "I am very sad",
    "Worst experience ever",
    "This is disappointing"
]

# 1 = Positive, 0 = Negative
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

# 2. Load BERT Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Tokenize Text
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# 4. Load Pre-trained BERT Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# 5. Prepare Labels
labels = torch.tensor(labels)

# 6. Train Model
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

print("Training BERT Model...")

model.train()

for epoch in range(3):
    optimizer.zero_grad()

    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        labels=labels
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

# 7. Predict New Text
def predict_sentiment(text):
    model.eval()

    tokens = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        output = model(**tokens)

    prediction = torch.argmax(output.logits, dim=1).item()

    return "Positive" if prediction == 1 else "Negative"

# 8. Test Prediction
test_text = "This movie is wonderful and amazing"

print("\nInput Text:", test_text)
print("Predicted Sentiment:", predict_sentiment(test_text))

# 9. Save Model
model.save_pretrained("./bert_model")
tokenizer.save_pretrained("./bert_model")

print("\nModel saved successfully!")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training BERT Model...
Epoch 1, Loss: 0.7228
Epoch 2, Loss: 0.6756
Epoch 3, Loss: 0.6001

Input Text: This movie is wonderful and amazing
Predicted Sentiment: Positive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
